In [3]:

import streamlit as st
import pandas as pd
import numpy as np
from io import BytesIO

st.set_page_config(
    page_title="Data Cleaning & Validation",
    page_icon="🧹",
    layout="wide"
)

st.title("🧹 Dataset Cleaning & Validation System")
st.write("Upload, clean, validate and download your dataset.")

# ---------------------------------
# 1. Upload Dataset
# ---------------------------------

uploaded_file = st.file_uploader(
    "Upload CSV or Excel Dataset",
    type=["csv", "xlsx"]
)

if uploaded_file is not None:

    if uploaded_file.name.endswith(".csv"):
        df = pd.read_csv(uploaded_file)
    else:
        df = pd.read_excel(uploaded_file)

    original_df = df.copy()

    st.success("Dataset uploaded successfully!")

    # ---------------------------------
    # 2. Dataset Overview
    # ---------------------------------

    st.header("1. Dataset Overview")

    col1, col2, col3 = st.columns(3)

    col1.metric("Total Rows", df.shape[0])
    col2.metric("Total Columns", df.shape[1])
    col3.metric("Duplicate Rows", df.duplicated().sum())

    st.subheader("Dataset Preview")
    st.dataframe(df.head(10), use_container_width=True)

    # ---------------------------------
    # 3. Assign Data Types
    # ---------------------------------

    st.header("2. Assign Data Types")

    st.write("Select each column and assign its required data type.")

    data_types = [
        "Keep Original",
        "String",
        "Integer",
        "Float",
        "Date",
        "Boolean",
        "Category"
    ]

    dtype_mapping = {}

    for column in df.columns:

        selected_type = st.selectbox(
            f"Data type for {column}",
            data_types,
            key=f"dtype_{column}"
        )

        dtype_mapping[column] = selected_type

    # ---------------------------------
    # 4. Apply Data Types
    # ---------------------------------

    if st.button("Apply Data Types"):

        converted_df = df.copy()
        errors = []

        for column, dtype in dtype_mapping.items():

            if dtype == "Keep Original":
                continue

            original = converted_df[column].copy()

            if dtype == "String":
                converted_df[column] = original.astype("string")

            elif dtype == "Integer":
                converted_df[column] = pd.to_numeric(
                    original, errors="coerce"
                ).astype("Int64")

            elif dtype == "Float":
                converted_df[column] = pd.to_numeric(
                    original, errors="coerce"
                )

            elif dtype == "Date":
                converted_df[column] = pd.to_datetime(
                    original, errors="coerce"
                )

            elif dtype == "Boolean":

                bool_map = {
                    "true": True, "false": False,
                    "1": True, "0": False,
                    "yes": True, "no": False,
                    "y": True, "n": False
                }

                converted_df[column] = (
                    original.astype("string")
                    .str.strip()
                    .str.lower()
                    .map(bool_map)
                    .astype("boolean")
                )

            elif dtype == "Category":
                converted_df[column] = original.astype("category")

            # Detect conversion errors
            invalid = (
                original.notna() &
                converted_df[column].isna()
            )

            for index in original[invalid].index:
                errors.append({
                    "Column": column,
                    "Row": index,
                    "Invalid Value": original.loc[index]
                })

        st.session_state["converted_df"] = converted_df
        st.session_state["conversion_errors"] = errors

        st.success("Data types applied successfully!")

    # ---------------------------------
    # 5. Validation and Cleaning
    # ---------------------------------

    if "converted_df" in st.session_state:

        clean_df = st.session_state["converted_df"].copy()

        st.header("3. Data Quality Report")

        # Missing values
        missing = clean_df.isnull().sum()

        # Duplicate values
        duplicates = clean_df.duplicated().sum()

        # Outlier detection using IQR
        outlier_report = {}

        for column in clean_df.select_dtypes(
            include=np.number
        ).columns:

            Q1 = clean_df[column].quantile(0.25)
            Q3 = clean_df[column].quantile(0.75)

            IQR = Q3 - Q1

            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR

            outliers = (
                (clean_df[column] < lower) |
                (clean_df[column] > upper)
            )

            outlier_report[column] = outliers.sum()

        report1, report2, report3 = st.columns(3)

        report1.metric(
            "Missing Values",
            int(missing.sum())
        )

        report2.metric(
            "Duplicate Rows",
            int(duplicates)
        )

        report3.metric(
            "Conversion Errors",
            len(st.session_state["conversion_errors"])
        )

        st.subheader("Missing Values Report")
        st.dataframe(
            missing.rename("Missing Count"),
            use_container_width=True
        )

        st.subheader("Outlier Report")
        st.dataframe(
            pd.DataFrame(
                list(outlier_report.items()),
                columns=["Column", "Outlier Count"]
            ),
            use_container_width=True
        )

        st.subheader("Data Type Conversion Errors")

        st.dataframe(
            pd.DataFrame(st.session_state["conversion_errors"]),
            use_container_width=True
        )

        # ---------------------------------
        # 6. Unique Column Selection
        # ---------------------------------

        st.header("4. Unique Value Validation")

        unique_columns = st.multiselect(
            "Select columns that must contain unique values",
            clean_df.columns.tolist()
        )

        unique_violations = 0

        if unique_columns:

            duplicate_unique = clean_df.duplicated(
                subset=unique_columns,
                keep=False
            )

            unique_violations = duplicate_unique.sum()

            st.write(
                "Rows violating unique constraints:",
                int(unique_violations)
            )

            if unique_violations > 0:
                st.dataframe(
                    clean_df[duplicate_unique],
                    use_container_width=True
                )

        # ---------------------------------
        # 7. Cleaning Options
        # ---------------------------------

        st.header("5. Clean Dataset")

        st.subheader("Handle Missing Values")

        missing_action = st.selectbox(
            "Choose missing value handling method",
            [
                "Keep Missing Values",
                "Remove Rows with Missing Values",
                "Fill Numerical with Median",
                "Fill Numerical with Mean",
                "Fill with Mode"
            ]
        )

        st.subheader("Handle Duplicates")

        remove_duplicates = st.checkbox(
            "Remove completely duplicated rows"
        )

        remove_unique_duplicates = st.checkbox(
            "Remove duplicate values based on selected unique columns"
        )

        st.subheader("Handle Outliers")

        remove_outliers = st.checkbox(
            "Remove rows containing numerical outliers"
        )

        # ---------------------------------
        # 8. Execute Cleaning
        # ---------------------------------

        if st.button("Run Data Cleaning"):

            result = clean_df.copy()

            # Missing values
            if missing_action == "Remove Rows with Missing Values":
                result = result.dropna()

            elif missing_action == "Fill Numerical with Median":

                for column in result.select_dtypes(
                    include=np.number
                ).columns:

                    result[column] = result[column].fillna(
                        result[column].median()
                    )

            elif missing_action == "Fill Numerical with Mean":

                for column in result.select_dtypes(
                    include=np.number
                ).columns:

                    result[column] = result[column].fillna(
                        result[column].mean()
                    )

            elif missing_action == "Fill with Mode":

                for column in result.columns:

                    modes = result[column].mode(
                        dropna=True
                    )

                    if not modes.empty:
                        result[column] = result[column].fillna(
                            modes.iloc[0]
                        )

            # Remove full-row duplicates
            if remove_duplicates:
                result = result.drop_duplicates()

            # Remove unique constraint violations
            if remove_unique_duplicates and unique_columns:
                result = result.drop_duplicates(
                    subset=unique_columns,
                    keep="first"
                )

            # Remove outliers
            if remove_outliers:

                for column in result.select_dtypes(
                    include=np.number
                ).columns:

                    Q1 = result[column].quantile(0.25)
                    Q3 = result[column].quantile(0.75)

                    IQR = Q3 - Q1

                    lower = Q1 - 1.5 * IQR
                    upper = Q3 + 1.5 * IQR

                    result = result[
                        (result[column].isna()) |
                        (
                            (result[column] >= lower) &
                            (result[column] <= upper)
                        )
                    ]

            st.session_state["final_df"] = result

            st.success("Dataset cleaning completed!")

        # ---------------------------------
        # 9. Final Dataset and Download
        # ---------------------------------

        if "final_df" in st.session_state:

            final_df = st.session_state["final_df"]

            st.header("6. Final Cleaned Dataset")

            a, b, c = st.columns(3)

            a.metric("Original Rows", len(original_df))
            b.metric("Cleaned Rows", len(final_df))
            c.metric(
                "Removed Rows",
                len(original_df) - len(final_df)
            )

            st.dataframe(
                final_df,
                use_container_width=True
            )

            # CSV Download
            csv = final_df.to_csv(index=False).encode("utf-8")

            st.download_button(
                label="Download Cleaned CSV",
                data=csv,
                file_name="cleaned_dataset.csv",
                mime="text/csv"
            )

else:
    st.info("Please upload a CSV or Excel file to begin.")


2026-11-23 21:18:20.022 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 21:18:20.039 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 21:18:22.990 
  command:

    streamlit run C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-11-23 21:18:22.993 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 21:18:23.002 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 21:18:23.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 21:18:23.007 Thread 'MainThread': missing ScriptRunContext! This warning can 